# Kaggle 07. DSPy Prompt Optimization for Logical Fallacy Classification

Цель ноутбука: проверить не ручной подбор prompt-ов, а каноничный DSPy-пайплайн:

1. `Signature`: `text_ru -> label`, где `label` строго одна из 9 меток COCOLOFA-RU v2.
2. Базовый модуль: `dspy.Predict`, без `ChainOfThought`, потому что предыдущий reasoning-прогон ухудшил `none`.
3. Оптимизация только на `dev`: сначала `LabeledFewShot`, затем `MIPROv2(auto="light")`.
4. Метрика оптимизации: weighted exact match с усилением редких классов и отдельным контролем false positive rate на `none`.
5. Финальная оценка: held-out `test`, который не используется при подборе prompt-а.

Ноутбук provider-agnostic. DSPy официально работает через `dspy.LM` / LiteLLM:
можно использовать API-провайдера или локальную модель, поднятую как OpenAI-compatible endpoint
через vLLM, SGLang, Ollama или другой сервер.

## Как подключить модель

Перед запуском ячеек задайте переменные окружения в Kaggle Secrets или прямо в ноутбуке.

Вариант API:

```python
import os
os.environ["DSPY_MODEL"] = "openai/gpt-5-mini"
os.environ["OPENAI_API_KEY"] = "..."
```

Вариант локального OpenAI-compatible endpoint:

```python
import os
os.environ["DSPY_MODEL"] = "openai/google/gemma-4-e4b-it"
os.environ["DSPY_API_BASE"] = "http://localhost:8000/v1"
os.environ["DSPY_API_KEY"] = "local"
```

Важно: сам DSPy-ноутбук не поднимает сервер модели. Это сделано намеренно:
DSPy-часть остается каноничной и не зависит от конкретного способа serving-а.

In [ ]:
%pip install -q -U "dspy>=3.0.0" litellm pandas numpy scikit-learn matplotlib seaborn tqdm

In [ ]:
import json
import logging
import os
import random
from collections import Counter
from pathlib import Path
from typing import Literal

import dspy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from tqdm.notebook import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("kaggle_dspy_prompt_optimization")
sns.set_theme(style="whitegrid")

In [ ]:
INPUT_JSONL_CANDIDATES = sorted(Path("/kaggle/input").rglob("*.jsonl"))
assert INPUT_JSONL_CANDIDATES, (
    "Upload cocolofa_ru_v2.jsonl, cocolofa_ru_v2_masked.jsonl, "
    "or cocolofa_ru_v2_explanations.jsonl to /kaggle/input first."
)

for path in INPUT_JSONL_CANDIDATES:
    print(path)

EXPLANATIONS_DATASET_PATH = next(
    (path for path in INPUT_JSONL_CANDIDATES if path.name == "cocolofa_ru_v2_explanations.jsonl"),
    None,
)
RAW_DATASET_PATH = next(
    (path for path in INPUT_JSONL_CANDIDATES if path.name == "cocolofa_ru_v2.jsonl"),
    None,
)
MASKED_DATASET_PATH = next(
    (path for path in INPUT_JSONL_CANDIDATES if path.name == "cocolofa_ru_v2_masked.jsonl"),
    None,
)
DATASET_PATH = EXPLANATIONS_DATASET_PATH or RAW_DATASET_PATH or MASKED_DATASET_PATH
assert DATASET_PATH is not None, "No supported COCOLOFA-RU dataset was found."

OUTPUT_ROOT = Path("/kaggle/working/phase3_dspy_prompt_optimization")
TEXT_COLUMN = "text_ru"
EVAL_SPLIT = "test"
SEED = 42

# Optimization budget. Keep this small for the first run.
OPT_TRAIN_PER_CLASS = 6
OPT_DEV_PER_CLASS = 8
MAX_TEST_ROWS = 100  # Set to None for the full held-out test split.

# DSPy optimizer settings.
LABELED_FEWSHOT_K = 9
RUN_BOOTSTRAP_FEWSHOT = False
RUN_MIPRO = True
MIPRO_AUTO = "light"
MIPRO_NUM_TRIALS = 6
MIPRO_MAX_BOOTSTRAPPED_DEMOS = 0
MIPRO_MAX_LABELED_DEMOS = 9
MIPRO_MINIBATCH_SIZE = 24
NUM_THREADS = 1

# LM settings. For local endpoints use DSPY_API_BASE + DSPY_MODEL="openai/<served-model-name>".
DSPY_MODEL = os.getenv("DSPY_MODEL", "openai/gpt-5-mini")
DSPY_API_BASE = os.getenv("DSPY_API_BASE")
DSPY_API_KEY = os.getenv("DSPY_API_KEY") or os.getenv("OPENAI_API_KEY") or os.getenv("GEMINI_API_KEY")
if DSPY_API_BASE and not DSPY_API_KEY:
    DSPY_API_KEY = "local"

LM_TEMPERATURE = float(os.getenv("DSPY_TEMPERATURE", "0.0"))
LM_MAX_TOKENS = int(os.getenv("DSPY_MAX_TOKENS", "64"))
ADAPTER_NAME = os.getenv("DSPY_ADAPTER", "chat").lower()  # chat is most compatible with local LMs.

{
    "dataset_path": str(DATASET_PATH),
    "output_root": str(OUTPUT_ROOT),
    "text_column": TEXT_COLUMN,
    "seed": SEED,
    "opt_train_per_class": OPT_TRAIN_PER_CLASS,
    "opt_dev_per_class": OPT_DEV_PER_CLASS,
    "max_test_rows": MAX_TEST_ROWS,
    "dspy_model": DSPY_MODEL,
    "dspy_api_base": DSPY_API_BASE,
    "adapter": ADAPTER_NAME,
    "run_mipro": RUN_MIPRO,
}

In [ ]:
LABELS = [
    "none",
    "appeal to authority",
    "appeal to majority",
    "appeal to nature",
    "appeal to tradition",
    "appeal to worse problems",
    "false dilemma",
    "hasty generalization",
    "slippery slope",
]
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}
ACCEPTED_TRANSLATION_STATUSES = {"ok", "repaired_ok"}

LABEL_DESCRIPTIONS_RU = {
    "none": "логической ошибки из списка нет",
    "appeal to authority": "вывод принимается из-за авторитета источника вместо аргументов",
    "appeal to majority": "вывод принимается из-за популярности мнения",
    "appeal to nature": "естественность или неестественность объявляется доказательством правильности",
    "appeal to tradition": "вывод обосновывается тем, что так принято или так было всегда",
    "appeal to worse problems": "проблема обесценивается ссылкой на более серьезные проблемы",
    "false dilemma": "ситуация искусственно сведена к двум вариантам",
    "hasty generalization": "общий вывод делается по недостаточному числу случаев",
    "slippery slope": "утверждается цепочка тяжелых последствий без достаточного обоснования",
}

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)


def normalize_label(value) -> str:
    return " ".join(str(value).strip().lower().replace("_", " ").replace("-", " ").split())


NORMALIZED_TO_LABEL = {normalize_label(label): label for label in LABELS}


def canonical_label(value) -> str | None:
    normalized = normalize_label(value)
    if normalized in NORMALIZED_TO_LABEL:
        return NORMALIZED_TO_LABEL[normalized]
    for label in LABELS:
        if normalize_label(label) in normalized:
            return label
    return None


def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False))
            handle.write("\n")


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def infer_label_str(row: pd.Series) -> str:
    if "label_str" in row and pd.notna(row["label_str"]):
        return str(row["label_str"])
    if "label" in row and pd.notna(row["label"]):
        return str(row["label"])
    if "label_id" in row and pd.notna(row["label_id"]):
        return ID_TO_LABEL[int(row["label_id"])]
    raise ValueError(f"Cannot infer label for row: {row.to_dict()}")


set_global_seed(SEED)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
raw_rows = load_jsonl(DATASET_PATH)
df = pd.DataFrame(raw_rows)
assert not df.empty, DATASET_PATH
assert "split" in df.columns, "Dataset must contain split column."

if TEXT_COLUMN not in df.columns:
    fallback = "text_masked" if "text_masked" in df.columns else "text"
    logger.warning("%s not found. Falling back to %s.", TEXT_COLUMN, fallback)
    TEXT_COLUMN = fallback

if "translation_status" in df.columns:
    before = len(df)
    df = df[df["translation_status"].isin(ACCEPTED_TRANSLATION_STATUSES)].copy()
    logger.info("Filtered translation statuses: %d -> %d", before, len(df))

df["label_str"] = df.apply(infer_label_str, axis=1)
df["label_str"] = df["label_str"].map(lambda value: canonical_label(value) or value)
bad_labels = sorted(set(df["label_str"]) - set(LABELS))
assert not bad_labels, f"Unknown labels: {bad_labels}"

df = df[df["split"].isin(["train", "dev", "test"])].copy()
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
df = df[df[TEXT_COLUMN].str.strip().ne("")].copy()

train_df = df[df["split"] == "train"].reset_index(drop=True)
dev_df = df[df["split"] == "dev"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
assert len(train_df) and len(dev_df) and len(test_df), df["split"].value_counts().to_dict()

display(pd.DataFrame({
    "split": ["train", "dev", "test"],
    "rows": [len(train_df), len(dev_df), len(test_df)],
}))
display(df["label_str"].value_counts().reindex(LABELS).rename("rows").to_frame())

In [ ]:
def stratified_per_class_sample(source_df: pd.DataFrame, per_class: int, seed: int) -> pd.DataFrame:
    parts = []
    for label in LABELS:
        label_df = source_df[source_df["label_str"] == label]
        assert len(label_df), f"No rows for label={label}"
        n = min(per_class, len(label_df))
        parts.append(label_df.sample(n=n, random_state=seed))
    sampled = pd.concat(parts, ignore_index=True)
    sampled["_label_order"] = sampled["label_str"].map(LABEL_TO_ID)
    sampled = sampled.sort_values(["_label_order"]).drop(columns=["_label_order"]).reset_index(drop=True)
    return sampled


def round_robin_by_label(source_df: pd.DataFrame) -> pd.DataFrame:
    groups = {
        label: source_df[source_df["label_str"] == label].reset_index(drop=True)
        for label in LABELS
    }
    ordered_rows = []
    max_len = max(len(group) for group in groups.values())
    for idx in range(max_len):
        for label in LABELS:
            group = groups[label]
            if idx < len(group):
                ordered_rows.append(group.iloc[idx])
    return pd.DataFrame(ordered_rows).reset_index(drop=True)


train_opt_df = round_robin_by_label(stratified_per_class_sample(train_df, OPT_TRAIN_PER_CLASS, SEED))
dev_opt_df = stratified_per_class_sample(dev_df, OPT_DEV_PER_CLASS, SEED)
test_eval_df = test_df.head(MAX_TEST_ROWS).copy() if MAX_TEST_ROWS is not None else test_df.copy()

print("Optimization train rows:", len(train_opt_df))
print("Optimization dev rows:", len(dev_opt_df))
print("Held-out test rows:", len(test_eval_df))
display(train_opt_df["label_str"].value_counts().reindex(LABELS).rename("train_opt").to_frame())
display(dev_opt_df["label_str"].value_counts().reindex(LABELS).rename("dev_opt").to_frame())
display(test_eval_df["label_str"].value_counts().reindex(LABELS).rename("test_eval").to_frame())

In [ ]:
lm_kwargs = {
    "temperature": LM_TEMPERATURE,
    "max_tokens": LM_MAX_TOKENS,
}
if DSPY_API_BASE:
    lm_kwargs["api_base"] = DSPY_API_BASE
if DSPY_API_KEY:
    lm_kwargs["api_key"] = DSPY_API_KEY

lm = dspy.LM(DSPY_MODEL, **lm_kwargs)

if ADAPTER_NAME == "json":
    adapter = dspy.JSONAdapter()
elif ADAPTER_NAME == "chat":
    adapter = dspy.ChatAdapter()
else:
    raise ValueError(f"Unsupported DSPY_ADAPTER={ADAPTER_NAME!r}. Use 'chat' or 'json'.")

dspy.configure(lm=lm, adapter=adapter)

print("DSPy version:", getattr(dspy, "__version__", "unknown"))
print("LM:", DSPY_MODEL)
print("API base:", DSPY_API_BASE)
print("Adapter:", type(adapter).__name__)

In [ ]:
LabelLiteral = Literal[
    "none",
    "appeal to authority",
    "appeal to majority",
    "appeal to nature",
    "appeal to tradition",
    "appeal to worse problems",
    "false dilemma",
    "hasty generalization",
    "slippery slope",
]


class FallacyLabelSignature(dspy.Signature):
    """Классифицируй русский аргумент в один класс логической ошибки.

    Используй только заданную таксономию. Если ошибка не выражена явно,
    выбирай `none`. Не используй знания об истинности фактов во внешнем мире:
    оценивай именно аргументативный переход от посылок к выводу.
    """

    text_ru: str = dspy.InputField(desc="Русский аргументативный текст.")
    label: LabelLiteral = dspy.OutputField(
        desc=(
            "Ровно одна метка из списка: "
            + ", ".join(LABELS)
            + ". Для корректных или недостаточно явно ошибочных рассуждений выбирай none."
        )
    )


def make_program() -> dspy.Predict:
    return dspy.Predict(FallacyLabelSignature)


def to_dspy_examples(source_df: pd.DataFrame) -> list[dspy.Example]:
    examples = []
    for row in source_df.itertuples(index=False):
        examples.append(
            dspy.Example(
                text_ru=str(getattr(row, TEXT_COLUMN)),
                label=str(row.label_str),
            ).with_inputs("text_ru")
        )
    return examples


trainset = to_dspy_examples(train_opt_df)
devset = to_dspy_examples(dev_opt_df)
testset = to_dspy_examples(test_eval_df)

print("DSPy examples:", {"trainset": len(trainset), "devset": len(devset), "testset": len(testset)})

In [ ]:
train_counts = train_df["label_str"].value_counts().to_dict()
raw_weights = {
    label: 1.0 / np.sqrt(max(train_counts.get(label, 1), 1))
    for label in LABELS
}
max_weight = max(raw_weights.values())
CLASS_WEIGHTS = {
    label: raw_weights[label] / max_weight
    for label in LABELS
}
# Keep the majority `none` class important enough to control false positives.
CLASS_WEIGHTS["none"] = max(CLASS_WEIGHTS["none"], 0.75)


def weighted_exact_match(example, pred, trace=None) -> float:
    gold = canonical_label(getattr(example, "label", None))
    predicted = canonical_label(getattr(pred, "label", None))
    if gold is None or predicted is None:
        return 0.0
    return float(CLASS_WEIGHTS.get(gold, 1.0)) if gold == predicted else 0.0


pd.DataFrame(
    [{"label": label, "train_count": train_counts.get(label, 0), "metric_weight": CLASS_WEIGHTS[label]} for label in LABELS]
)

In [ ]:
def compute_metrics(rows: list[dict]) -> dict:
    y_true = [row["gold_label"] for row in rows]
    y_pred = [row["pred_label"] for row in rows]
    valid_predictions = [pred in LABELS for pred in y_pred]

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=LABELS,
        zero_division=0,
    )
    per_class = {
        label: {
            "precision": float(precision[idx]),
            "recall": float(recall[idx]),
            "f1": float(f1[idx]),
            "support": int(support[idx]),
        }
        for idx, label in enumerate(LABELS)
    }

    none_total = sum(1 for gold in y_true if gold == "none")
    none_false_positive = sum(1 for gold, pred in zip(y_true, y_pred) if gold == "none" and pred != "none")
    weighted_denominator = sum(CLASS_WEIGHTS.get(gold, 1.0) for gold in y_true)
    weighted_numerator = sum(
        CLASS_WEIGHTS.get(gold, 1.0)
        for gold, pred in zip(y_true, y_pred)
        if gold == pred
    )

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(np.mean(precision)),
        "macro_recall": float(np.mean(recall)),
        "macro_f1": float(np.mean(f1)),
        "weighted_exact_match": float(weighted_numerator / weighted_denominator) if weighted_denominator else 0.0,
        "none_false_positive_rate": float(none_false_positive / none_total) if none_total else 0.0,
        "invalid_label_rate": float(1.0 - np.mean(valid_predictions)) if valid_predictions else 0.0,
        "per_class": per_class,
    }


def evaluate_program(program, eval_df: pd.DataFrame, name: str, output_dir: Path) -> tuple[dict, list[dict]]:
    rows = []
    for row in tqdm(list(eval_df.itertuples(index=False)), desc=f"Evaluate {name}"):
        sample_id = getattr(row, "sample_id", None)
        text = str(getattr(row, TEXT_COLUMN))
        gold_label = str(row.label_str)

        try:
            pred = program(text_ru=text)
            raw_label = getattr(pred, "label", None)
            error = None
        except Exception as exc:
            pred = None
            raw_label = None
            error = repr(exc)

        pred_label = canonical_label(raw_label) or "__invalid__"
        rows.append(
            {
                "program": name,
                "split": getattr(row, "split"),
                "sample_id": int(sample_id) if sample_id is not None and not pd.isna(sample_id) else None,
                "gold_label": gold_label,
                "pred_label": pred_label,
                "raw_label": str(raw_label),
                "parse_ok": pred_label in LABELS,
                "text_column": TEXT_COLUMN,
                "text": text,
                "error": error,
            }
        )

    metrics = compute_metrics(rows)
    output_dir.mkdir(parents=True, exist_ok=True)
    write_json(output_dir / f"{name}_metrics.json", metrics)
    write_jsonl(output_dir / f"{name}_predictions.jsonl", rows)
    write_json(
        output_dir / f"{name}_classification_report.json",
        classification_report(
            [row["gold_label"] for row in rows],
            [row["pred_label"] for row in rows],
            labels=LABELS,
            output_dict=True,
            zero_division=0,
        ),
    )
    write_json(
        output_dir / f"{name}_confusion_matrix.json",
        {
            "labels": LABELS,
            "matrix": confusion_matrix(
                [row["gold_label"] for row in rows],
                [row["pred_label"] for row in rows],
                labels=LABELS,
            ).tolist(),
        },
    )
    return metrics, rows


def summarize_metrics(metrics_by_program: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for name, metrics in metrics_by_program.items():
        rows.append({
            "program": name,
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "weighted_exact_match": metrics["weighted_exact_match"],
            "none_false_positive_rate": metrics["none_false_positive_rate"],
            "invalid_label_rate": metrics["invalid_label_rate"],
        })
    return pd.DataFrame(rows).sort_values(["weighted_exact_match", "macro_f1"], ascending=False)

In [ ]:
config_payload = {
    "dataset_path": str(DATASET_PATH),
    "output_root": str(OUTPUT_ROOT),
    "text_column": TEXT_COLUMN,
    "seed": SEED,
    "labels": LABELS,
    "class_weights": CLASS_WEIGHTS,
    "opt_train_per_class": OPT_TRAIN_PER_CLASS,
    "opt_dev_per_class": OPT_DEV_PER_CLASS,
    "max_test_rows": MAX_TEST_ROWS,
    "dspy_model": DSPY_MODEL,
    "dspy_api_base": DSPY_API_BASE,
    "adapter": ADAPTER_NAME,
    "lm_temperature": LM_TEMPERATURE,
    "lm_max_tokens": LM_MAX_TOKENS,
    "labeled_fewshot_k": LABELED_FEWSHOT_K,
    "run_bootstrap_fewshot": RUN_BOOTSTRAP_FEWSHOT,
    "run_mipro": RUN_MIPRO,
    "mipro_auto": MIPRO_AUTO,
    "mipro_num_trials": MIPRO_NUM_TRIALS,
    "mipro_max_bootstrapped_demos": MIPRO_MAX_BOOTSTRAPPED_DEMOS,
    "mipro_max_labeled_demos": MIPRO_MAX_LABELED_DEMOS,
}
write_json(OUTPUT_ROOT / "config.json", config_payload)
config_payload

In [ ]:
programs = {}
dev_metrics = {}

baseline_program = make_program()
programs["predict_zero_shot"] = baseline_program
dev_metrics["predict_zero_shot"], _ = evaluate_program(
    baseline_program,
    dev_opt_df,
    "dev_predict_zero_shot",
    OUTPUT_ROOT / "dev",
)

labeled_program = dspy.LabeledFewShot(k=LABELED_FEWSHOT_K).compile(
    make_program(),
    trainset=trainset,
)
programs["labeled_fewshot"] = labeled_program
dev_metrics["labeled_fewshot"], _ = evaluate_program(
    labeled_program,
    dev_opt_df,
    "dev_labeled_fewshot",
    OUTPUT_ROOT / "dev",
)

summarize_metrics(dev_metrics)

In [ ]:
if RUN_BOOTSTRAP_FEWSHOT:
    bootstrap_optimizer = dspy.BootstrapFewShot(
        metric=weighted_exact_match,
        max_bootstrapped_demos=2,
        max_labeled_demos=LABELED_FEWSHOT_K,
        max_rounds=1,
        max_errors=20,
    )
    bootstrap_program = bootstrap_optimizer.compile(make_program(), trainset=trainset)
    programs["bootstrap_fewshot"] = bootstrap_program
    dev_metrics["bootstrap_fewshot"], _ = evaluate_program(
        bootstrap_program,
        dev_opt_df,
        "dev_bootstrap_fewshot",
        OUTPUT_ROOT / "dev",
    )
else:
    print("BootstrapFewShot skipped. Set RUN_BOOTSTRAP_FEWSHOT=True to enable it.")

summarize_metrics(dev_metrics)

In [ ]:
if RUN_MIPRO:
    mipro_optimizer = dspy.MIPROv2(
        metric=weighted_exact_match,
        auto=MIPRO_AUTO,
        num_threads=NUM_THREADS,
        max_errors=20,
        verbose=True,
    )
    (OUTPUT_ROOT / "mipro_logs").mkdir(parents=True, exist_ok=True)
    mipro_program = mipro_optimizer.compile(
        make_program(),
        trainset=trainset,
        valset=devset,
        num_trials=MIPRO_NUM_TRIALS,
        max_bootstrapped_demos=MIPRO_MAX_BOOTSTRAPPED_DEMOS,
        max_labeled_demos=MIPRO_MAX_LABELED_DEMOS,
        minibatch=True,
        minibatch_size=min(MIPRO_MINIBATCH_SIZE, len(devset)),
        minibatch_full_eval_steps=2,
        seed=SEED,
    )
    programs["mipro_light"] = mipro_program
    dev_metrics["mipro_light"], _ = evaluate_program(
        mipro_program,
        dev_opt_df,
        "dev_mipro_light",
        OUTPUT_ROOT / "dev",
    )
else:
    print("MIPROv2 skipped. Set RUN_MIPRO=True to enable it.")

dev_summary_df = summarize_metrics(dev_metrics)
display(dev_summary_df)
write_json(OUTPUT_ROOT / "dev_summary_metrics.json", dev_metrics)
dev_summary_df.to_csv(OUTPUT_ROOT / "dev_summary_metrics.csv", index=False)

In [ ]:
best_program_name = summarize_metrics(dev_metrics).iloc[0]["program"]
best_program = programs[best_program_name]
print("Best program selected on dev:", best_program_name)

test_metrics = {}
test_rows_by_program = {}
for name, program in programs.items():
    metrics, rows = evaluate_program(
        program,
        test_eval_df,
        f"test_{name}",
        OUTPUT_ROOT / "test",
    )
    test_metrics[name] = metrics
    test_rows_by_program[name] = rows

test_summary_df = summarize_metrics(test_metrics)
display(test_summary_df)
write_json(OUTPUT_ROOT / "test_summary_metrics.json", test_metrics)
test_summary_df.to_csv(OUTPUT_ROOT / "test_summary_metrics.csv", index=False)

try:
    best_program.save(str(OUTPUT_ROOT / f"best_program_{best_program_name}.json"))
except Exception as exc:
    logger.warning("Could not save DSPy program: %r", exc)

In [ ]:
best_test_rows = test_rows_by_program[best_program_name]
best_cm = confusion_matrix(
    [row["gold_label"] for row in best_test_rows],
    [row["pred_label"] for row in best_test_rows],
    labels=LABELS,
)

plt.figure(figsize=(11, 9))
sns.heatmap(best_cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABELS, yticklabels=LABELS)
plt.title(f"Confusion matrix: {best_program_name}")
plt.xlabel("Predicted label")
plt.ylabel("Gold label")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / f"confusion_matrix_{best_program_name}.png", dpi=180)
plt.show()

per_class_df = pd.DataFrame(test_metrics[best_program_name]["per_class"]).T.reset_index(names="label")
display(per_class_df.sort_values("f1", ascending=False))

In [ ]:
reference_prompt_results = pd.DataFrame([
    {"program": "Gemma 4 E4B zero-shot manual", "accuracy": 0.660, "macro_f1": 0.676},
    {"program": "Gemma 4 E4B few-shot manual", "accuracy": 0.760, "macro_f1": 0.743},
    {"program": "Gemma 4 E4B reasoning manual", "accuracy": 0.600, "macro_f1": 0.639},
])

comparison_df = pd.concat(
    [
        test_summary_df[["program", "accuracy", "macro_f1"]].assign(source="DSPy current run"),
        reference_prompt_results.assign(source="previous manual prompt run"),
    ],
    ignore_index=True,
)
display(comparison_df.sort_values("macro_f1", ascending=False))

## Интерпретация результатов

Для научного отчета фиксируем три уровня вывода:

1. `predict_zero_shot` показывает, насколько DSPy signature сама по себе улучшает или ухудшает ручной zero-shot prompt.
2. `labeled_fewshot` проверяет эффект правильно выбранных демонстраций без дополнительного поиска.
3. `mipro_light` проверяет, может ли DSPy автоматически подобрать более удачную инструкцию и набор examples на `dev`.

Важное ограничение: если `MAX_TEST_ROWS = 100`, это быстрый smoke benchmark, а не финальная оценка.
Для итоговой таблицы ВКР нужно поставить `MAX_TEST_ROWS = None` и запустить held-out test целиком.